[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/susom/tide2-core/blob/main/notebooks/tide2_pipeline.ipynb)

# TIDE 2.0 De-identification Pipeline

End-to-end clinical note de-identification using the TIDE 2.0 toolkit.

The pipeline runs three stages via Ray:

1. **Transformer** — NER inference (GPU-accelerated when available, CPU fallback for Mac)
2. **Recognizer** — CPU-parallelized Presidio recognition (combines transformer results with regex recognizers and patient known values)
3. **Anonymizer** — CPU-parallelized Presidio anonymization (HIPS cryptographic replacement)

Each stage can be toggled on/off. When a stage is skipped, its output from a previous run is expected on disk, enabling checkpoint/restart workflows.

**Data sources supported:**
- Local text files (default — uses bundled `sample_data/`)
- BigQuery tables
- Any Pandas DataFrame with a `note_text` column

**Platform support:**
- **GPU (Linux/CUDA)**: Full performance with GPU-accelerated transformer inference
- **CPU-only (Mac/Linux)**: Automatic fallback to CPU inference (slower but functional)

## Running in Google Colab

To run this notebook in [Google Colab](https://colab.research.google.com/):

1. Click **Runtime → Run all**.
2. The first code cell installs `tide2` via `uv` and downloads the bundled
   sample data (a no-op when running locally).
3. On first run, the transformer model
   (`StanfordAIMI/stanford-deidentifier-v2`, ~500&nbsp;MB) downloads from the
   HuggingFace Hub — this can take a few minutes.
4. The default runtime is **CPU-only**. For faster transformer inference, pick a
   GPU **before** running: **Runtime → Change runtime type → T4 GPU → Save**. The
   free-tier **T4** is plenty for this pipeline. The accelerator can only be
   chosen from this menu — there's no notebook code or URL setting for it — but
   once selected the pipeline auto-detects and uses it (`num_gpus = None` below).
   The setup cell prints which GPU, if any, was detected.

> The Streamlit visualizer is for **local use only** and is not practical inside
> Colab; the JSON path-printing cells still work.

In [ ]:
# =============================================================================
# Google Colab setup (no-op when running locally)
# =============================================================================
import sys

IN_COLAB = "google.colab" in sys.modules

# Pin tide2 and the sample-data source to the same release for reproducibility.
TIDE2_VERSION = "1.1.0"

if IN_COLAB:
    # Colab's default kernel is cpython-3.12.x, which matches tide2's
    # >=3.12,<3.13 constraint. Use an explicit check (not `assert`, which `python
    # -O` would strip) so a future Colab image change fails loudly.
    if sys.version_info[:2] != (3, 12):
        raise RuntimeError(
            f"tide2 requires Python 3.12; this Colab kernel is "
            f"{sys.version.split()[0]}. Use the default Colab runtime (3.12)."
        )

    # uv is pre-installed on Colab. Only install when tide2 is missing or the
    # version differs, to avoid a slow reinstall (and dep churn) on re-runs.
    import importlib.metadata as _md

    try:
        _installed = _md.version("tide2")
    except _md.PackageNotFoundError:
        _installed = None

    if _installed != TIDE2_VERSION:
        !uv pip install --system -q tide2=={TIDE2_VERSION}
    else:
        print(f"tide2=={TIDE2_VERSION} already installed — skipping install")

    # Fetch ONLY the needed sample_data files from GitHub raw (no full clone).
    # Pin to the matching release tag so the download is reproducible.
    import urllib.request
    from pathlib import Path

    RAW = f"https://raw.githubusercontent.com/susom/tide2-core/v{TIDE2_VERSION}/notebooks/sample_data"
    NOTE_IDS = [f"note_{i}" for i in range(1, 14)]  # note_1 .. note_13

    base = Path("/content/sample_data")
    (base / "text_files").mkdir(parents=True, exist_ok=True)
    (base / "patient_phi").mkdir(parents=True, exist_ok=True)

    def _fetch(rel_path):
        dest = base / rel_path
        if not dest.exists():
            urllib.request.urlretrieve(f"{RAW}/{rel_path}", dest)  # noqa: S310

    for note_id in NOTE_IDS:
        _fetch(f"text_files/{note_id}.txt")
        _fetch(f"patient_phi/{note_id}.json")

    print(f"✅ Colab setup complete — {len(NOTE_IDS)} notes fetched")

    # Report the accelerator so it's clear whether a GPU (e.g. the free-tier T4)
    # was selected. The pipeline auto-detects and uses it; pick one via
    # Runtime → Change runtime type if this prints "CPU only".
    try:
        import torch

        if torch.cuda.is_available():
            print(f"🎮 GPU detected: {torch.cuda.get_device_name(0)}")
        else:
            print("🖥️  CPU only — for a GPU: Runtime → Change runtime type → T4 GPU")
    except ImportError:
        pass

In [ ]:
import json
import logging
from pathlib import Path

import pandas as pd

from tide2.runner import LocalJobRunner

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## Configuration

Configuration is **auto-tuned based on platform**:
- Mac: CPU-only mode with conservative resource usage
- Linux/GPU: Full GPU acceleration with higher throughput settings

You can override any setting below if needed.

In [ ]:
import multiprocessing
import platform

# --- GCP project (optional) ---
# Only needed for BigQuery or GCS model downloads. The transformer model
# downloads from HuggingFace Hub by default, so GCP credentials are NOT required.
# Uncomment and set these if you want to use GCS-hosted model weights:
# project_id = "your-gcp-project-id"
# bucket_name = "your-gcs-bucket"
# os.environ["GCLOUD_PROJECT"] = project_id
project_id = ""
bucket_name = ""

# Transformer model (downloaded from HuggingFace Hub when bucket_name is None)
model_name = "StanfordAIMI/stanford-deidentifier-v2"

# --- Platform detection ---
is_mac = platform.system() == "Darwin"
available_cpus = multiprocessing.cpu_count()

# Detect GPUs (guarded — torch may be absent or CUDA unavailable on CPU runtimes)
num_detected_gpus = 0
try:
    import torch

    num_detected_gpus = torch.cuda.device_count()
except Exception:
    num_detected_gpus = 0
has_gpu = num_detected_gpus > 0

if is_mac:
    print(f"🍎 Running on Mac ({available_cpus} CPUs) — using CPU-only mode")
else:
    print(f"🐧 Running on {platform.system()} ({available_cpus} CPUs)")
print(f"   GPUs detected: {num_detected_gpus}")

# --- Ray configuration (auto-tuned for platform) ---
# GPU settings: None = auto-detect (0 on Mac, available GPUs on Linux)
num_gpus = None

# Transformer actors: None = auto-detect (num_gpus on GPU, ~25% CPUs on Mac)
# Set explicitly to control parallelism for transformer inference
num_transformer_actors = None

# CPU actors for recognizer/anonymizer: scale based on available cores
num_actors = max(1, available_cpus // 4) if is_mac else max(2, available_cpus // 3)

# Batch sizes: smaller on CPU for better memory efficiency
cpu_batch_size = 50 if is_mac else 100
transformer_batch_size = 16 if is_mac else 512  # Ray Data batch size for transformer

# Object store: size from *available* (free) RAM on the CPU/Mac branch, with a
# sane clamp. The GPU/Linux value is left as-is (Ray auto-sizes from total RAM).
object_store_gb = 4
if is_mac:  # CPU branch only — do not change the GPU/Linux value
    import psutil

    available_gb = psutil.virtual_memory().available / 1024**3
    # Use ~25% of *available* RAM, clamped to a practical range.
    object_store_gb = max(1, min(8, round(available_gb * 0.25)))

print(f"   Recognizer/Anonymizer actors: {num_actors}")
print(f"   Transformer actors: {num_transformer_actors or 'auto'}")
print(f"   CPU batch size: {cpu_batch_size}, Object store: {object_store_gb}GB")

# --- Small-box budgeting (e.g. 2-CPU Google Colab) ---------------------------
# WHY THIS EXISTS — the Colab deadlock, in full:
#
# Ray Data's streaming executor runs every operator of a stage concurrently
# (read -> flat_map -> transformer actor -> BIO actor -> write). Under Ray 2.55's
# ReservationOpResourceAllocator it must reserve a *minimum* CPU slice for each
# eligible operator AT THE SAME TIME. If those minimums sum to more than the
# cluster's CPUs, nothing schedules and the stage hangs forever at 0/1
# (logs show "backpressured:tasks(ResourceBudget)", "Resources: 0.2 CPU"). On a
# 2-CPU Colab box this is exactly what happens. There are TWO independent causes,
# and BOTH must be addressed (verified empirically — fixing only one still hangs):
#
#   1. Whole-CPU operator reservations. The defaults reserve ~1 CPU per operator;
#      read + flat_map + actor + agg + write > 2. FIX: fractional CPUs (below) so
#      the concurrent sum fits within `available_cpus`.
#   2. The checkpoint shuffle. Row-level resume injects sort + repartition ops
#      (Sort Sample / Shuffle Map / Shuffle Reduce / Repartition), adding several
#      more eligible operators that re-trigger the deadlock EVEN WITH fractional
#      CPUs. FIX: enable_checkpoint=False (loses resume, not correctness).
#
# How to choose values for ANY hardware (C = total CPUs, G = total GPUs):
#   * Big box (C >= ~16): use library defaults — leave both dicts empty. The
#     library auto-scales actor counts and whole-CPU reservations fit fine.
#   * Small box (C <= 4): set fractional CPUs + enable_checkpoint=False. Keep the
#     sum of each stage's CONCURRENT operator reservations <= C:
#       - Transformer = read + flat_map + write + actors*actor_cpu + agg*agg_cpu
#       - Recognizer/Anonymizer = read + actors*(num_cpus + worker_num_cpus) + write
#     GPU mode: the transformer actor is GPU-pinned (0 CPU), so actor_cpu ~ 0.
#     CPU mode: the actor needs ~1 CPU AND transformer_cpus caps its torch
#     threads, so give it ~(C-1) for usable single-box throughput.
#
# We assemble `transformer_extra` / `cpu_stage_extra` kwargs dicts that are EMPTY
# on large boxes (library defaults apply unchanged) and populated on small boxes.
# We never pass None to float params: keys are simply omitted on large boxes.
SMALL_BOX_MAX_CPUS = 4  # <=4-CPU clusters (e.g. 2-CPU Colab) need the fractional-CPU budget below
small_box = available_cpus <= SMALL_BOX_MAX_CPUS
transformer_extra: dict = {}
cpu_stage_extra: dict = {}

if small_box:
    # Transformer stage: read + flat_map + write + actors*actor_cpu + agg*agg_cpu
    transformer_extra = {
        "read_cpus": 0.25,
        "flat_map_cpus": 0.25,
        "write_cpus": 0.25,
        "num_transformer_actors": 1,
        "num_agg_actors": 1,
        "enable_checkpoint": False,  # cause #2: checkpoint shuffle deadlocks tiny clusters
    }
    if has_gpu:
        # GPU actor is GPU-pinned (0 CPU); transformer_cpus is a small optional floor.
        # C=2,G=1: 0.25+0.25+0.25 + 1*0(GPU actor) + 1*0.5(agg) = 1.25 <= 2 ✓
        transformer_extra["transformer_cpus"] = 0.25
        transformer_extra["agg_num_cpus"] = 0.5
    else:
        # CPU actor needs ~1 CPU; transformer_cpus also caps torch threads, so give
        # it ~all-but-one core. C=2,G=0: 0.25+0.25+0.25 + 1*1.0(actor) + 1*0.25(agg) = 2.0 <= 2 ✓
        transformer_extra["transformer_cpus"] = max(0.5, available_cpus - 1.0)
        transformer_extra["agg_num_cpus"] = 0.25

    # Recognizer/anonymizer: read + actors*(supervisor_cpu + worker_cpu) + write
    # C=2: 0.25(read) + 1*(0.5+1.0) + 0.25(write) = 2.0 <= 2 ✓
    cpu_stage_extra = {
        "num_actors": 1,
        "num_cpus": 0.5,
        "worker_num_cpus": 1.0,
        "read_cpus": 0.25,
        "write_cpus": 0.25,
        "enable_checkpoint": False,  # cause #2 again, for the CPU stages
    }

    print(
        f"\n⚙️  Small box ({available_cpus} CPUs, "
        f"{'GPU' if has_gpu else 'CPU-only'}) — applying fractional-CPU budget + "
        f"disabling checkpointing to avoid the Ray Data deadlock:"
    )
    print(f"   transformer_extra = {transformer_extra}")
    print(f"   cpu_stage_extra   = {cpu_stage_extra}")
    print("   ℹ️  Checkpointing (row-level resume) is OFF here — re-running starts from scratch.")
    if not has_gpu:
        print("   ⚠️  CPU-only inference will be SLOW (single-threaded) — this restores correctness, not speed.")
else:
    print(
        f"\n⚙️  Large box ({available_cpus} CPUs) — using library defaults (whole-CPU reservations + checkpointing on)."
    )
    print(
        "   Rule of thumb if you tune manually: keep the sum of a stage's CONCURRENT\n"
        "   operator CPU reservations <= total CPUs. Transformer stage =\n"
        "   read + flat_map + write + actors*actor_cpu + agg*agg_cpu; recognizer/\n"
        "   anonymizer = read + actors*(num_cpus + worker_num_cpus) + write.\n"
        "   On <=4-CPU boxes ALSO pass enable_checkpoint=False (the checkpoint\n"
        "   shuffle deadlocks the reservation allocator)."
    )

# --- Pipeline stage flags ---
# Set to False to skip a stage (will use existing output from a prior run)
run_transformer = True
run_recognizer = True
run_anonymizer = True

# --- HIPS keys for anonymization ---
# Dev/test keys (use real keys in production):
salt_hex = "00" * 32
key_hex = "11" * 32

# To load production keys from files, uncomment:
# salt_hex = open("/path/to/salt.key").read().strip()
# key_hex = open("/path/to/key.key").read().strip()

# --- Accession number hash configuration ---
acc_num_salt = "notebook_salt"
acc_num_study_id = "notebook_study_001"

## Data Loading

Choose one of the data sources below. The pipeline expects a DataFrame with at least a `note_text` column. Optional columns: `patient_id`, `patient_identifiers` (JSON string of PHI for known-value recognizers), `text_hash`.

In [ ]:
# =============================================================================
# OPTION 1: Local text files (default)
# =============================================================================
# Each .txt file is one clinical note; optional patient_phi/<note_id>.json
# files provide known PHI identifiers for the known-value recognizer.

# Resolve sample_data path — works whether the notebook is launched from the repo
# root (jupyter notebook notebooks/tide2_pipeline.ipynb), from the notebooks/
# directory, or on Colab (where the setup cell fetches data to /content).
# A candidate counts only if it actually contains note .txt files, so an empty
# or partially-created directory (e.g. /content/sample_data before the setup cell
# populates it) is skipped instead of causing a confusing "Loaded 0 notes" later.
candidates = [Path("sample_data"), Path("notebooks") / "sample_data", Path("/content/sample_data")]
sample_data_dir = next((p for p in candidates if any((p / "text_files").glob("*.txt"))), None)
if sample_data_dir is None:
    raise FileNotFoundError(
        "No sample_data/ with text_files/*.txt found. Please run this notebook "
        "from the repository root directory, from the notebooks/ directory, or "
        "on Colab (run the setup cell first)."
    )

text_files_dir = sample_data_dir / "text_files"
patient_phi_dir = sample_data_dir / "patient_phi"

records = []
for txt_file in sorted(text_files_dir.glob("*.txt")):
    note_id = txt_file.stem
    note_text = txt_file.read_text(encoding="utf-8")

    phi_file = patient_phi_dir / f"{note_id}.json"
    patient_phi = {}
    if phi_file.exists():
        with phi_file.open(encoding="utf-8") as f:
            patient_phi = json.load(f)

    records.append(
        {
            "note_text": note_text,
            "patient_id": note_id,
            "patient_identifiers": json.dumps(patient_phi),
        }
    )

df = pd.DataFrame(records)
output_dir = str(sample_data_dir / "pipeline_output")

print(f"Loaded {len(df)} notes from {text_files_dir}")
assert len(df) > 0, "No notes found"

In [ ]:
# =============================================================================
# OPTION 2: BigQuery table (uncomment this cell and skip Option 1)
# =============================================================================
# from google.cloud import bigquery
#
# bq_table = "project.dataset.table_name"
# output_dir = "/data/tide2_data/bq_output"
#
# client = bigquery.Client(project=project_id)
# df = client.query(f"SELECT * FROM `{bq_table}`").to_dataframe()
# print(f"Loaded {len(df)} notes from {bq_table}")

## Run Pipeline

Initialize the `LocalJobRunner` and run the full de-identification pipeline. The `run_pipeline()` method handles all intermediate Parquet files, transformer chunk reassembly, and stage sequencing internally.

**Platform-specific optimizations:**
- **Mac (CPU-only)**: Smaller batch sizes, fewer actors to avoid memory pressure
- **GPU systems**: Larger batches for throughput, more parallel actors

Intermediate files are written to `output_dir` following this layout:
```
output_dir/
├── 01_transformer_input.parquet
├── 02_transformer_output/
├── 03_recognizer_input.parquet
├── 04_recognizer_output/
├── 05_anonymizer_input.parquet
├── 06_anonymizer_output/
├── cli_recognizer_json/      # if produce_visualizer_json=True
└── cli_anonymizer_json/      # if produce_visualizer_json=True
```

In [ ]:
runner = LocalJobRunner(
    num_cpus=None,  # let Ray auto-detect all logical CPUs
    num_gpus=num_gpus,
    object_store_gb=object_store_gb,
)

try:
    result = runner.run_pipeline(
        input_data=df,
        output_dir=output_dir,
        model_name=model_name,
        run_transformer=run_transformer,
        run_recognizer=run_recognizer,
        run_anonymizer=run_anonymizer,
        produce_visualizer_json=True,
        salt_hex=salt_hex,
        key_hex=key_hex,
        # transformer_extra / cpu_stage_extra are empty on large boxes (library
        # defaults apply) and carry the fractional-CPU budget on small boxes.
        transformer_kwargs={
            "bucket_name": bucket_name,
            "project_id": project_id,
            "num_gpus": num_gpus,
            "num_transformer_actors": num_transformer_actors,
            "batch_size": transformer_batch_size,
            **transformer_extra,
        },
        recognizer_kwargs={
            "num_actors": num_actors,
            "batch_size": cpu_batch_size,
            **cpu_stage_extra,
        },
        anonymizer_kwargs={
            "num_actors": num_actors,
            "batch_size": cpu_batch_size,
            "acc_num_salt": acc_num_salt,
            "acc_num_study_id": acc_num_study_id,
            "jitter_required": False,
            **cpu_stage_extra,
        },
    )
finally:
    runner.shutdown()

## Pipeline Results

In [ ]:
print("=" * 60)
print("PIPELINE RESULTS")
print("=" * 60)
for key, value in result.items():
    if isinstance(value, dict):
        print(f"\n  {key}:")
        for k, v in value.items():
            print(f"    {k}: {v}")
    else:
        print(f"  {key}: {value}")

## Verify Anonymized Output

Read and display a few rows from the anonymizer output to confirm the pipeline ran correctly.

In [ ]:
import pyarrow.parquet as pq

output_path = Path(output_dir)
anonymizer_output_dir = output_path / "06_anonymizer_output"
anonymizer_files = list(anonymizer_output_dir.glob("**/*.parquet"))

if anonymizer_files:
    print(f"Reading {len(anonymizer_files)} anonymizer output file(s)")
    for pf in anonymizer_files:
        df_output = pq.read_table(pf).to_pandas()
        print(f"\nFile: {pf.name}  |  Rows: {len(df_output)}  |  Columns: {list(df_output.columns)}")

        for idx, row in df_output.head(3).iterrows():
            text_hash = str(row.get("text_hash", "N/A"))[:16] + "..."
            note_text = row.get("anonymized_note_text", "") or row.get("note_text", "") or ""
            print(f"\n  Record {idx}:")
            print(f"    text_hash: {text_hash}")
            if note_text:
                print(f"    note_text (anonymized): {note_text[:100]}...")
            print(f"    entity_count: {row.get('entity_count', 0)}")
else:
    print("No anonymizer output found — run with run_anonymizer=True")

## Create Evaluation Files (Optional)

Generate Parquet files for downstream evaluation scripts:
- `texts.parquet` — text_id, text_content
- `predicted_spans.parquet` — text_id, span_start, span_end, span_tag
- `ml_spans.parquet` — note_id, span_start, span_end, span_tag, score, recognizer_name

In [ ]:
output_path = Path(output_dir)
recognizer_output_dir = output_path / "04_recognizer_output"
transformer_input_path = output_path / "01_transformer_input.parquet"

rec_files = list(recognizer_output_dir.glob("**/*.parquet"))
if not rec_files:
    print("No recognizer output found — skipping evaluation files")
else:
    # Merge recognizer output with note_text from transformer input
    dfs_rec = [pq.read_table(pf).to_pandas() for pf in rec_files]
    df_rec = pd.concat(dfs_rec, ignore_index=True)
    df_input = pd.read_parquet(transformer_input_path)
    df_merged = df_rec.merge(df_input[["text_hash", "note_text", "patient_id"]], on="text_hash", how="left")

    # --- texts.parquet ---
    df_texts = (
        df_merged[["patient_id", "note_text"]]
        .rename(columns={"patient_id": "text_id", "note_text": "text_content"})
        .drop_duplicates(subset=["text_id"])
    )
    df_texts.to_parquet(output_path / "texts.parquet", index=False)
    print(f"texts.parquet: {len(df_texts)} rows")

    # --- predicted_spans.parquet and ml_spans.parquet ---
    predicted_records = []
    ml_records = []
    for _, row in df_merged.iterrows():
        text_id = str(row.get("patient_id", row.get("text_hash", "unknown")))
        results_json = row.get("recognizer_results_json", "[]")
        if not results_json or results_json == "[]":
            continue
        try:
            results = json.loads(results_json)
        except json.JSONDecodeError:
            continue
        for ent in results:
            predicted_records.append(
                {
                    "text_id": text_id,
                    "span_start": ent.get("start"),
                    "span_end": ent.get("end"),
                    "span_tag": ent.get("entity_type"),
                }
            )
            ml_records.append(
                {
                    "note_id": text_id,
                    "span_start": ent.get("start"),
                    "span_end": ent.get("end"),
                    "span_tag": ent.get("entity_type"),
                    "score": ent.get("score", 0.0),
                    "recognizer_name": ent.get("recognition_metadata", {}).get("recognizer_name", "unknown"),
                }
            )

    df_predicted = pd.DataFrame(predicted_records)
    df_predicted.to_parquet(output_path / "predicted_spans.parquet", index=False)
    print(f"predicted_spans.parquet: {len(df_predicted)} rows")

    df_ml = pd.DataFrame(ml_records)
    df_ml.to_parquet(output_path / "ml_spans.parquet", index=False)
    print(f"ml_spans.parquet: {len(df_ml)} rows")

    if len(df_ml) > 0:
        print("\nSpan tag distribution:")
        print(df_ml["span_tag"].value_counts().to_string())

## Launch Visualizer (Optional)

The pipeline wrote JSON files to `cli_recognizer_json/` and `cli_anonymizer_json/` (when `produce_visualizer_json=True`). 

> **Colab note:** the Streamlit visualizer is for **local use only** and is not
> practical inside Colab. The path-printing cell below still works.

**Launch from command line**
```bash
tide2-visualizer
```

**Run the cells below** to see the data paths

In [ ]:
# Print absolute paths for the visualizer data directories
# Uses the output_dir from the pipeline result to ensure paths match actual output

# Get the actual output directory from pipeline result (or fall back to output_dir variable)
actual_output_dir = result.get("output_dir", output_dir) if "result" in dir() else output_dir
output_path = Path(actual_output_dir).resolve()  # Get absolute path

cli_rec = output_path / "cli_recognizer_json"
cli_anon = output_path / "cli_anonymizer_json"

rec_count = len(list(cli_rec.glob("*.json"))) if cli_rec.exists() else 0
anon_count = len(list(cli_anon.glob("*.json"))) if cli_anon.exists() else 0

print("=" * 70)
print("VISUALIZER DATA DIRECTORIES (copy these paths)")
print("=" * 70)
print(f"\nRecognizer JSON ({rec_count} files):")
print(f"{cli_rec}")
print(f"\nAnonymizer JSON ({anon_count} files):")
print(f"{cli_anon}")
print("\n" + "=" * 70)

if rec_count == 0 and anon_count == 0:
    print("\n⚠️  No visualizer JSON files found.")
    print("   Re-run the pipeline with produce_visualizer_json=True")
else:
    print("\n✅ Copy the paths above into the Streamlit visualizer when prompted.")